# 25 · 和 DeepRare 比：同一个模型，两种 agent 设计

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → 19 基因工具 ✅ → 20 大考 ✅ → 21 正式评测 ✅ → 22 做成服务 ✅ → 23 和别人比 ✅ → 24 和大模型比 ✅ → [25 和 DeepRare 比 ✅]
```

> **这一章的代码在合并请求 #27 里**（和第 26、27 章同一个）。DeepRare 考 220 道题（开发 20 + 留出 200），DeepSeek 约 ¥9；宽松判分不到 ¥0.1。运行这本笔记不花钱：DeepRare 的每个回答都存成了文件，结果从文件和 MLflow 里读。

## 背景：最直接的对手

第 23 章比了传统工具 LIRICAL，第 24 章比了三家最强的大模型。还差一个：**另一个罕见病诊断 agent**。

**DeepRare** 是一个罕见病诊断 agent（Zhao 等，2025 年预印本、2026 年发表在 *Nature*，代码开源），和我们做的是同一件事：给症状，出诊断。它是这个项目最直接的对手，也是总体计划第一天就写下的对照组（B4）。

两个 agent 的设计思路很不一样：

| | DeepRare | 我们的 agent |
|---|---|---|
| 思路 | **什么都查一遍**：上网搜、查好几个外部网站、读网页、查病例库，层层汇总 | **查该查的**：本机的工具，模型决定查什么、查几次 |
| 知识从哪来 | 互联网 + 外部网站 + 病例库 | 本机固定版本的 HPO、已发表病人 |
| 每道题 | 约 28 次模型调用，几分钟 | 4–8 次，约 13 秒 |
| 能不能遮住病人自己的论文 | **不能**（它会上网搜） | 能 |

这一章回答：**同一个大模型，换成 DeepRare 的设计，答得更准吗？**

**公司里的做法**：和"同类最强方案"在同一份题、同一个模型下比，叫**对标**（benchmark against the state of the art）。模型相同，比出来的差别才是设计的差别。

## 这一章做的 6 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 把 DeepRare 跑起来 | 它自己的环境；一个包装器；记录它读到的一切 | 不改它的提示词和流程，只做必须的改动 |
| 2 | 省钱、省时的验证 | 先跑开发题；查询向量预先算好；每道题的回答存成文件 | 不在不该花的地方花钱 |
| 3 | 先写方案 | 预注册；两个不对等事先写明 | 结果可信 |
| 4 | 为什么这么慢 | 拆开看一道题的时间 | 速度也是比较的一部分 |
| 5 | 结果 | 严格判分、宽松判分（DeepRare 作者的判法）、逐题配对 | 回答方案里的问题 |
| 6 | 它读到了病人自己的论文吗 | 在它读过的所有内容里找论文编号、DOI、标题 | 量出"不遮挡"占了多少便宜 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
import logging, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # 一条和我们无关的提示
logging.getLogger("mlflow").setLevel(logging.WARNING)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 把 DeepRare 跑起来

### 在干什么
把 DeepRare 的代码（`third_party/DeepRare`，提交 `7897820`）在本机跑通，让它考和我们同样的题。

### 背景
DeepRare 原文用 DeepSeek-R1、OpenAI 的模型、6 个来源的病例库（5 个没有公开）、还要谷歌或必应的搜索密钥。完全照搬做不到，也没必要：**我们要比的是设计，不是模型**。所以只做必须的改动，每一处都写进方案：

| | 原文 | 这里 | 为什么 |
|---|---|---|---|
| 大模型 | DeepSeek-R1 | **`deepseek-flash`，不思考**，和我们的 agent 同一个 | 比设计，不比模型 |
| 病例库 | 6 个来源 | 只有公开的 PubMed 病例（约 1 万例） | 其他 5 个没公开 |
| 网页搜索 | 谷歌 / 必应 / DuckDuckGo | DuckDuckGo | 不需要密钥 |
| 代码 | | 修了一处 bug（搜索工具参数传错位置）；相似病例检索改成一次算完（结果相同） | 不修跑不起来 |

### 用了什么技术，为什么
- DeepRare 用**它自己的 Python 环境**（`third_party/DeepRare/.venv`），我们的代码通过一个**子进程**调用它：两边的依赖互不干扰
- `src/raredx/baselines/deeprare.py` 是包装器：把一道题交给 DeepRare，拿回它的答案，把病名对应到 OMIM 编号，交给我们的评测流程判分——**和其他所有系统走同一套评测**
- **记录它读到的一切**：每道题它发给模型的全部内容、找到的相似病例、查到的工具结果，都存下来。第 6 步要用

### 公司里怎么做
把别人的系统**原样**接进自己的评测框架（不改它的核心逻辑），叫**适配器**（adapter）。所有改动列清单，是复现别人工作的基本规矩。

---

## 第 2 步 · 省钱、省时的验证

### 在干什么
在花大钱、跑大半夜之前，先确认每一环都能用，而且出了故障不白花钱。

### 背景
DeepRare 一道题几分钟、几十次模型调用，还要连好几个外部网站。200 道题跑一晚上，中途断网、网站超时都很正常。要是跑完才发现哪里错了，钱和时间都白花了。

### 用了什么技术，为什么
四件事，每件都是踩坑以后加的：

1. **先考开发题**：先 3 道，再 20 道（ci-20）。3 道里跑完的那道：178 秒、28 次模型调用、约 ¥0.06——量出真实单价，再决定敢不敢跑 200 道
2. **查询向量预先算好**：DeepRare 找相似病例时，要用 OpenAI 的向量模型把症状变成向量。这一步要连 OpenRouter，而本机网络时不时会被重置连接。所以 220 道题的查询向量**一次算好存起来**（`data/processed/deeprare/query-embeddings.json`，不到 $0.01），考试时不再依赖它
3. **网络故障不算答错**：一开始，外部网站超时导致 DeepRare 没给出答案，被算成"答错"。这不公平——那是网络的问题，不是 DeepRare 的问题。改成：接口故障的题记为"丢失"，只补问丢失的题（`--fill-in`，第 24 章加的）。同时把并发从 3 降到 2：开 3 道同时跑时，外部网站开始超时，20 道丢了 16 道
4. **每道题的回答立刻存成文件**（`data/processed/deeprare/answers/`）：万一进程挂了、MLflow 出问题，已经答完的题不用重问、不用重花钱。重跑时先看文件里有没有，有就直接读回来

### 公司里怎么做
长时间、按次付费的批处理任务，标准做法就是**检查点**（checkpointing）和**幂等**（idempotent：重跑一次和跑一次结果一样）。外部依赖不稳定时，**把故障和失败分开记**，否则统计的是网络质量，不是系统质量。

---

## 第 3 步 · 先写方案

### 在干什么
DeepRare 碰留出集之前，写好方案提交：[`docs/comparison_plan_deeprare.md`](../comparison_plan_deeprare.md)。

### 背景
和前几章一样：先写下怎么判，再考。这一次有两个绕不开的不对等，必须事先写明。

### 两个不对等

**1. DeepRare 没法遮挡。** 它会上网搜，会查 PubCaseFinder、PhenoBrain、病例库，这些地方都可能有病人自己的那篇论文。第 23 章量过，不遮挡能让分数高出 30 多个百分点。**这个不对等对 DeepRare 有利。** 所以要记下它读到的一切，考完逐题检查它有没有读到病人自己的论文（第 6 步）。

**2. DeepRare 只给病名，不给编号。** 有两种判分，都报：
- **严格**：病名对应到 OMIM 编号（模糊匹配，相似度 ≥ 90），编号对才算对
- **宽松**（DeepRare 作者自己的判法）：把前 5 个病名和确诊病名交给大模型，问"有没有、排第几"。**我们的 agent 也用同一个判法判一遍**，一视同仁

**开发题上看到的情况，考前补进了方案**：DeepRare 20 道全答了，但它给的 100 个病名里只有 45 个能按名字对上 OMIM 编号。它习惯写"Tapasin Deficiency (TAPBP Deficiency)"这样的名字，严格判法对不上就算错，哪怕说的就是确诊的病。所以**严格判分会明显低估 DeepRare**，报告时把宽松判分放在同样显眼的位置。

### 公司里怎么做
评测口径不同时（一个给编号、一个给名字），**同时报两种口径，并用同一把尺子量所有系统**。只挑对自己有利的口径，评审一眼就能看出来。

---

## 第 4 步 · 为什么这么慢

### 在干什么
拆开一道题，看时间花在哪。

### 背景
DeepRare 200 道题跑了一晚上，我们的 agent 一道约 13 秒。这个差别本身就是比较的一部分：医生等 13 秒和等 5 分钟，是完全不同的产品。

### 结果
小考里那道题（178 秒，算快的）：

| 时间花在哪 | 多少 |
|---|---|
| 调用大模型 28 次，一次接一次 | 86 秒 |
| 上网搜索、用浏览器打开网页、查 PubCaseFinder / PhenoBrain / PubMed 等、加载它自己的模型 | 92 秒 |

28 次模型调用里，18 次是"总结一篇网页或摘要"（每搜到一篇都让模型读一遍），10 次是主流程（先给 5 个候选病，再逐个去各网站查资料、核对，没把握就再来一轮）。

我们的 agent 快，是因为：只调 4–8 次模型；所有工具都在本机内存里算，不上网；同一轮要查的几样东西同时查，不排队。

### 公司里怎么做
**延迟和成本是产品指标，和准确率一起报。** 一个准确率高 2 个百分点、但慢 20 倍的系统，在很多场景里是不能用的。

---

## 第 5 步 · 结果

### 在干什么
DeepRare 考完留出集的 200 道题（2026-10-14 18:49 到 10-15 01:14），按方案的两种判法判分，和 agent 逐题比。

### 背景
200 道**全部答出**，没有一道因为网络故障丢失——第 2 步加的那几道保险（并发降到 2、查询向量预先算好、故障不算答错）都起了作用。

### 执行
逐题结果已经存在 `docs/comparison_llm_hits.tsv`（严格判分）和 `docs/deeprare_lenient.json`（宽松判分）里，下面只是读出来算（不花钱）：

In [ ]:
import csv, json
from raredx.evals.scoring import p_value, sign_test, wilson
rows = list(csv.DictReader(open("docs/comparison_llm_hits.tsv"), delimiter="\t"))
def rate(s, k):
    n = sum(r[f"{s}@{k}"] == "1" for r in rows); lo, hi = wilson(n, len(rows))
    return f"{n / len(rows):6.1%} [{lo:.1%}–{hi:.1%}]"
def paired(a, b, k):
    w = sum(r[f"{a}@{k}"] == "1" and r[f"{b}@{k}"] == "0" for r in rows)
    l = sum(r[f"{b}@{k}"] == "1" and r[f"{a}@{k}"] == "0" for r in rows)
    return f"{w} : {l}，{p_value(sign_test(w, l))}"
print("严格判分（病名对上 OMIM 编号才算对）")
for s, name in (("agent", "最终 agent"), ("deeprare", "DeepRare")):
    print(f"  {name:<10} 第 1 名 {rate(s, 1)}   前 5 名 {rate(s, 5)}")
print(f"  逐题 agent : DeepRare   第 1 名 {paired('agent', 'deeprare', 1)}   前 5 名 {paired('agent', 'deeprare', 5)}")
lenient = json.load(open("docs/deeprare_lenient.json"))
print("\n宽松判分（DeepRare 作者的判法，大模型当裁判，两边都判）")
for k in (1, 5):
    hit = lambda x, s: x[s] is not None and x[s] <= k
    d, a = sum(hit(x, "deeprare") for x in lenient), sum(hit(x, "agent") for x in lenient)
    w = sum(hit(x, "agent") and not hit(x, "deeprare") for x in lenient)
    l = sum(hit(x, "deeprare") and not hit(x, "agent") for x in lenient)
    print(f"  前 {k} 名   agent {a / len(lenient):.1%}   DeepRare {d / len(lenient):.1%}   逐题 {w} : {l}，{p_value(sign_test(w, l))}")

### 结果怎么看

| 留出集 200 道 | 判法 | 第 1 名 | 前 5 名 | 逐题（agent 赢 : DeepRare 赢） |
|---|---|---|---|---|
| **最终 agent**（遮挡） | 严格 | **43.0%** | **58.5%** | 第 1 名 71 : 2，前 5 名 96 : 2 |
| DeepRare（不遮挡） | 严格 | 8.5% | 11.5% | |
| **最终 agent** | 宽松 | **42.5%** | **59.0%** | 第 1 名 48 : 18，前 5 名 62 : 14 |
| DeepRare | 宽松 | 27.5% | 35.0% | |

所有 p 值都小于 0.001。

- **主要结论成立，而且用对 DeepRare 最有利的判法也成立**：宽松判分下 DeepRare 升到 27.5%，agent 还是 42.5%
- **为什么严格和宽松差这么多**：DeepRare 的 1,000 个病名，只有 57% 能按名字对上 OMIM 编号（"Tapasin Deficiency (TAPBP Deficiency)" 这种写法）。所以严格判分对它不公平，方案里事先写了要把宽松判分放在同样显眼的位置
- **裁判靠不靠得住**：agent 的宽松分（42.5%）和严格分（43.0%）几乎一样——大模型裁判和按编号判，结论一致
- **我预计 DeepRare 严格 20%–30%、宽松 35%–45%**，实际都更低

**花费和速度**（按同一个价格算）：

| 每道题 | 模型调用 | 用时（中位数） | 花费 |
|---|---|---|---|
| DeepRare | 约 27 次 | 194 秒 | $0.0136 |
| 最终 agent | 4–8 次 | 13 秒 | $0.0056 |

**DeepRare 更贵（2.4 倍）、更慢（14 倍）、也更不准。**

**要说清楚的一点**：DeepRare 原文的分数比这里高得多。条件不同：原文用推理模型 DeepSeek-R1，这里是不思考的 `deepseek-flash`；病例库只有公开的部分；题目是遮挡后更难的留出集。所以结论是"**同一个便宜模型上，我们的设计更好**"，不是说原文的数字不对。

### 公司里怎么做
对标别人的系统时，**统一模型、统一题目、统一判法**，再把所有不对等之处列出来。结论的范围要和条件一致：这里能说"同一个模型下设计更好"，不能说"我们比 DeepRare 强"。

---

## 第 6 步 · 它读到病人自己的论文了吗

### 在干什么
DeepRare 不遮挡，可能在网上、病例库里读到病人自己那篇论文。把它每道题读过的所有内容（发给模型的全部文字、找到的相似病例）翻一遍，找病人论文的 PubMed 编号、PMC 编号、DOI 和标题。

### 背景
第 23 章量过：不遮挡能让 LIRICAL 高 33 个百分点。如果 DeepRare 大量读到了答案，它的分数就虚高；对比就不公平——对它有利的不公平。

### 执行

In [ ]:
exposure = json.load(open("docs/deeprare_exposure.json"))
seen = [c for c, signs in exposure.items() if any(signs.values())]
print(f"{len(seen)} 道 / {len(exposure)} 道读到了病人自己的论文")
print("按线索：", {k: sum(s[k] for s in exposure.values()) for k in ("pmid", "pmc", "doi", "title")})
by_case = {r["case_id"]: r for r in rows}
print("这几道 DeepRare 第 1 名答对：", sum(by_case[c]["deeprare@1"] == "1" for c in seen))

### 结果怎么看
- **200 道里只有 3 道**（1.5%）：都是它找到的相似病例里，正好有病人自己那篇论文（按 PMC 编号认出来）。它发给模型的文字里，没有一道出现论文的 PubMed 编号、DOI 或标题
- 这 3 道它第 1 名**都没答对**
- 我预计 20% 以上，实际远低于预期。一个可能的原因：它的病例库只有公开的 1 万个病例，未必收了这些论文——这一点没有逐一核对，只是推测

所以这次"不遮挡"几乎没给 DeepRare 占到便宜，结论不受这个不对等的影响。

**一个细节**：DeepRare 病例库的编号是 PMC 编号，PubMed 编号和 PMC 编号都是 7、8 位数字。找 PubMed 编号时如果也搜病例库的编号列表，碰巧数字相同就会误报。所以找 PubMed 编号时跳过了病例库编号，只用 PMC 编号去对。

### 公司里怎么做
评测里**防泄露要能证明**：不只是说"应该没泄露"，而是把系统读过的东西翻出来查，给出数字。

---

## 这一章做完，项目多了什么

| | |
|---|---|
| 新的对照 | DeepRare：同一个模型、同一份 200 道题 |
| 结论 | 严格和宽松两种判法，agent 都明显领先；更便宜、快 14 倍 |
| 新代码 | DeepRare 的包装器和运行器；每道题回答存文件、重跑读回 |
| 花费 | DeepSeek 约 ¥8（200 道）+ 开发题，宽松判分不到 ¥0.1 |

## 一句话

**同一个模型，"查该查的"胜过"什么都查一遍"：更准、更快、更便宜。**

## 面试题

1. **How does your agent compare with DeepRare?**
   > "Same model for both — deepseek-flash — on the same 200 held-out cases. Scored by OMIM id, the agent gets the confirmed disease first in 43% against DeepRare's 8.5%, 71 cases to 2 paired; with DeepRare's own LLM-judge scoring, applied to both, 42.5% against 27.5%, 48 to 18. DeepRare takes about 27 model calls and three minutes a case; the agent 4 to 8 calls and 13 seconds."

2. **Isn't that unfair to DeepRare?**
   > "Partly, and I wrote down how before the run: it used a non-reasoning model instead of R1, only the public part of its case bank, and it couldn't be masked — which favours it, so I searched everything it read for the patient's own paper: 3 cases of 200. The claim is about the design on the same cheap model, not about its published numbers."

**下一章 26 · 读病历**：让 agent 读得懂医生写的字。